# Transformer 网络

欢迎开始第 4 周的作业！这是深度学习专项课程中第 5 门课程的最后一份作业，也是整个专项课程的最后一份作业。恭喜你坚持到这里，即将完成全部学习！

在本课程的前面，你已经实现了顺序神经网络，例如 RNN、GRU 和 LSTM。在本 Notebook 中，你将探索 Transformer 架构，这是一种利用并行处理的神经网络，可让你大幅加快训练过程。 

**在此作业之后，你将能够**：

* 创建位置编码以捕获数据中的顺序关系
* 使用词嵌入计算缩放点积自注意力
* 实现带掩码的多头注意力
* 构建并训练 Transformer 模型

最后一次，让我们开始吧！

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/nlp-sequence-models/supplement/X39s5/optional-downloading-your-notebook-downloading-your-workspace-and-refreshing) 获取一份新的作业。

## 目录

- [依赖库](#0)
- [1 - 位置编码](#1)
    - [1.1 - 正弦角和余弦角](#1-1)
        - [练习 1 - get_angles](#ex-1)
    - [1.2 - 正弦和余弦位置编码](#1-2)
        - [练习 2 - positional_encoding](#ex-2)
- [2 - 掩码机制](#2)
    - [2.1 - 填充掩码](#2-1)
    - [2.2 - 前瞻掩码](#2-2)
- [3 - 自注意力](#3)
    - [练习 3 - scaled_dot_product_attention](#ex-3)
- [4 - 编码器](#4)
    - [4.1 编码器层](#4-1)
        - [练习 4 - EncoderLayer](#ex-4)
    - [4.2 - 完整编码器](#4-2)
        - [练习 5 - Encoder](#ex-5)
- [5 - 解码器](#5)
    - [5.1 - 解码器层](#5-1)
        - [练习 6 - DecoderLayer](#ex-6)
    - [5.2 - 完整解码器](#5-2)
        - [练习 7 - Decoder](#ex-7)
- [6 - Transformer](#6)
    - [练习 8 - Transformer](#ex-8)
- [7 - 参考文献](#7)

<a name='0'></a>
## 依赖库

运行以下单元格来加载你需要的包。

In [ ]:
### v1.6

In [ ]:
import tensorflow as tf
import time
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.layers import Embedding, MultiHeadAttention, Dense, Input, Dropout, LayerNormalization

<a name='1'></a>
## 1 - 位置编码

在序列任务中，元素的相对顺序对含义至关重要。RNN 等序列神经网络按顺序接收输入，因此顺序信息自然进入模型。Transformer 使用多头注意力并行处理输入，能够加快训练，但需要额外提供位置信息。位置编码通过以下正弦和余弦公式表示各输入元素的位置：
    
$$
PE_{(pos, 2i)}= sin\left(\frac{pos}{{10000}^{\frac{2i}{d}}}\right)
\tag{1}$$
<br>
$$
PE_{(pos, 2i+1)}= cos\left(\frac{pos}{{10000}^{\frac{2i}{d}}}\right)
\tag{2}$$

* $d$ 是词嵌入与位置编码共同的维度。
* $pos$ 是单词的位置。
* $k$ 指的是位置编码中的每个不同维度，其中 $i$ 等于 $k$ $//$ $2$。

为了培养一些关于位置编码的直觉，你可以将它们广泛地视为包含有关单词相对位置信息的特征。位置编码和词嵌入的总和最终是输入模型的内容。如果你只是对位置进行硬编码，例如将 1 或整数的矩阵添加到词嵌入，则语义会被扭曲。相反，正弦和余弦方程的值足够小（在 -1 和 1 之间），因此当你将位置编码添加到词嵌入时，词嵌入不会明显失真，而是富含位置信息。使用这两个方程的组合可以帮助你的 Transformer 网络关注输入数据的相对位置。这是关于位置编码的简短讨论，但为了进一步加深直觉，请查看*位置编码不计分实验室*。 

**注意：** 课堂讲解使用列向量，而本作业使用行向量，因此矩阵乘法的排列方式也相应调整。

<a name='1-1'></a>
### 1.1 - 正弦角和余弦角

请注意，即使正弦和余弦位置编码方程采用不同的参数（`2i` 与 `2i+1`，或偶数与奇数），两个方程的内部项是相同的：$$\theta(pos, i, d) = \frac{pos}{10000^{\frac{2i}{d}}} \tag{3}$$

在计算序列中单词的位置编码时考虑内部项。<br> 
$PE_{(pos, 0)}= sin\left(\frac{pos}{{10000}^{\frac{0}{d}}}\right)$，因为求解 `2i = 0` 给出 `i = 0` <br>
$PE_{(pos, 1)}= cos\left(\frac{pos}{{10000}^{\frac{0}{d}}}\right)$，因为求解 `2i + 1 = 1` 给出 `i = 0`

两者的角度相同！ $PE_{(pos, 2)}$ 和 $PE_{(pos, 3)}$ 的角度也相同，因为对于两者来说，都是 `i = 1`，因此内项是 $\left(\frac{pos}{{10000}^{\frac{2}{d}}}\right)$。此关系适用于所有成对的正弦和余弦曲线：

|      k         | <code>       0      </code>|<code>       1      </code>|<code>       2      </code>|<code>       3      </code>| <code> ... </code> |<code>      d - 2     </code>|<code>      d - 1     </code>| 
| ---------------- | :------: | ----------------- | ----------------- | ----------------- | ----- | ----------------- | ----------------- |
| encoding(0) = |[$sin(\theta(0, 0, d))$| $cos(\theta(0, 0, d))$| $sin(\theta(0, 1, d))$| $cos(\theta(0, 1, d))$|... |$sin(\theta(0, d//2, d))$| $cos(\theta(0, d//2, d))$]|
| encoding(1) = | [$sin(\theta(1, 0, d))$| $cos(\theta(1, 0, d))$| $sin(\theta(1, 1, d))$| $cos(\theta(1, 1, d))$|... |$sin(\theta(1, d//2, d))$| $cos(\theta(1, d//2, d))$]|
...
| encoding(pos) = | [$sin(\theta(pos, 0, d))$| $cos(\theta(pos, 0, d))$| $sin(\theta(pos, 1, d))$| $cos(\theta(pos, 1, d))$|... |$sin(\theta(pos, d//2, d))$| $cos(\theta(pos, d//2, d))]$|


<a name='ex-1'></a>
### 练习 1 - get_angles

实现函数 `get_angles()` 来计算正弦和余弦位置编码的可能角度

**提示**

- 如果 `k = [0, 1, 2, 3, 4, 5]`，那么，`i` 一定是 `i = [0, 0, 1, 1, 2, 2]`
- `i = k//2`

In [ ]:
# UNQ_C1 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION get_angles
def get_angles(pos, k, d):
    """
    获取位置编码的角度
    
    参数：
        pos -- 包含位置 [[0], [1], ...,[N-1]] 的列向量
        k --   包含维度跨度 [[0, 1, 2, ..., d-1]] 的行向量
        d(integer) -- 编码大小
    
    返回：
        angles -- (pos, d) NumPy 数组 
    """
    
    # START CODE HERE
    # 从维度跨度 k 获取 i
    i = None
    # 使用 pos、i 和 d 计算角度
    angles = None
    # END CODE HERE
    
    return angles

In [ ]:
from public_tests import *

get_angles_test(get_angles)

# 示例
position = 4
d_model = 8
pos_m = np.arange(position)[:, np.newaxis]
dims = np.arange(d_model)[np.newaxis, :]
get_angles(pos_m, dims, d_model)

<a name='1-2'></a>
### 1.2 - 正弦和余弦位置编码

现在你可以使用计算出的角度来计算正弦和余弦位置编码。

$$
PE_{(pos, 2i)}= sin\left(\frac{pos}{{10000}^{\frac{2i}{d}}}\right)
$$
<br>
$$
PE_{(pos, 2i+1)}= cos\left(\frac{pos}{{10000}^{\frac{2i}{d}}}\right)
$$

<a name='ex-2'></a>
### 练习 2 - positional_encoding

实现函数 `positional_encoding()` 来计算正弦和余弦位置编码

**提醒：** $i$ 为偶数时使用正弦方程，$i$ 为奇数时使用余弦方程。

#### 附加提示
* 你可能会发现 
[np.newaxis](https://numpy.org/doc/stable/user/basics.indexing.html#dimensional-indexing-tools) 有用取决于你选择的实现。 

In [ ]:
# UNQ_C2 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION positional_encoding
def positional_encoding(positions, d):
    """
    使用所有位置编码预先计算矩阵 
    
    参数：
        Positions (int) -- 要编码的最大位置数 
        d (int) -- 编码大小 
    
    返回：
        pos_encoding -- (1, position, d_model) 具有位置编码的矩阵
    """
    # START CODE HERE
    # 初始化所有角度的矩阵 angle_rads 
    angle_rads = get_angles(None,
                            None,
                            None)
  
    # 将 sin 应用于数组中的偶数索引； 2i
    angle_rads[:, 0::2] = None
  
    # 将 cos 应用于数组中的奇数索引； 2i+1
    angle_rads[:, 1::2] = None
    # END CODE HERE
    
    pos_encoding = angle_rads[np.newaxis, ...]
    
    return tf.cast(pos_encoding, dtype=tf.float32)

In [ ]:
# 单元测试    
positional_encoding_test(positional_encoding, get_angles)

计算位置编码干得好！现在你可以想象它们了。

In [ ]:
pos_encoding = positional_encoding(50, 512)

print (pos_encoding.shape)

plt.pcolormesh(pos_encoding[0], cmap='RdBu')
plt.xlabel('d')
plt.xlim((0, 512))
plt.ylabel('Position')
plt.colorbar()
plt.show()

每行代表一个位置编码 - 请注意没有任何行是相同的！你已经为每个单词创建了唯一的位置编码。

<a name='2'></a>
## 2 - 掩码机制

在构建 Transformer 网络时，有两种类型的掩码非常有用：*填充掩码*和*前瞻掩码*。两者都有助于 softmax 计算为输入句子中的单词提供适当的权重。 

<a name='2-1'></a>
### 2.1 - 填充掩码

通常，你的输入序列会超出网络可以处理的序列的最大长度。假设模型的最大长度为 5，则输入以下序列：

    [[“做”, “你”, “知道”, “何时”, “简”, “是”, “去”, “去”, “访问”, “非洲”], 
     [“简”, “访问”, “非洲”, “在”, “九月”],
     [“令人兴奋”，“！”]
    ]

可能会得到向量化的：

    [[ 71, 121, 4, 56, 99, 2344, 345, 1284, 15],
     [ 56, 1285, 15, 181, 545],
     [ 87, 600]
    ]
    
将序列传递到 Transformer 模型时，重要的是它们的长度一致。你可以通过用零填充序列并截断超过模型最大长度的句子来实现此目的：

    [[ 71, 121, 4, 56, 99],
     [ 2344, 345, 1284, 15, 0],
     [ 56, 1285, 15, 181, 545],
     [ 87, 600, 0, 0, 0],
    ]
    
长度超过 5 的序列会被截断，短于 5 的序列会补 0。由于填充值会影响 softmax，需要构造填充掩码，标明哪些位置应参与注意力计算（1），哪些位置应被忽略（0）。随后，在 softmax 之前将被屏蔽位置的注意力分数设为很大的负数（-1e9），使其权重接近 0。下面已提供实现，请仔细阅读，以便在模型中正确使用。

掩码作用于注意力分数。例如，若分数为 `[87, 600, 0, 0, 0]`，则屏蔽填充位置后可表示为 `[87, 600, -1e9, -1e9, -1e9]`；这样 softmax 就不会给填充位置分配有效权重。

Keras 中实现的 [MultiheadAttention](https://keras.io/api/layers/attention_layers/multi_head_attention/) 层使用此屏蔽逻辑。

**注意：** 下面的函数仅创建 _已填充序列_ 的掩码。本周晚些时候，你将完成一些有关 Transformer 应用程序的实验，其中将向你介绍 [TensorFlow tokenizer](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/text/Tokenizer) 和 [Hugging Face tokenizer](https://huggingface.co/docs/tokenizers/api/tokenizer)，它们在内部处理输入序列的填充（和截断）。

In [ ]:
def create_padding_mask(decoder_token_ids):
    """
    为填充单元创建矩阵掩码
    
    参数：
        decoder_token_ids -- (n, m) 矩阵
    
    返回：
        mask -- (n, 1, m) 二进制张量
    """    
    seq = 1 - tf.cast(tf.math.equal(decoder_token_ids, 0), tf.float32)
  
    # 添加额外尺寸以添加填充
    # 加到注意力 logits 上。
    # 这将允许稍后在比较序列时使用广播
    return seq[:, tf.newaxis, :] 

In [ ]:
x = tf.constant([[7., 6., 0., 0., 1.], [1., 2., 3., 0., 0.], [0., 0., 0., 4., 5.]])
print(create_padding_mask(x))

如果我们将 (1 - mask) 乘以 -1e9 并将其添加到样本输入序列中，则零点实际上会设置为负无穷大。请注意原始序列和屏蔽序列的 softmax 的区别：

In [ ]:
print(tf.keras.activations.softmax(x))
print(tf.keras.activations.softmax(x + (1 - create_padding_mask(x)) * -1.0e9))

<a name='2-2'></a>
### 2.2 - 前瞻掩码

训练时可以访问完整的正确输出，但预测当前位置时不应看到后续答案。前瞻掩码会屏蔽未来位置，让模型*不访问未来信息*，只根据已经给出的部分预测下一个输出。

例如，如果预期的正确输出是 `[1, 2, 3]`，并且你想查看模型是否正确预测了第一个值，是否可以预测第二个值，你将屏蔽第二个和第三个值。因此，你可以输入屏蔽序列 `[1, -1e9, -1e9]` 并查看它是否可以生成 `[1, 2, -1e9]`。

下面也提供了前瞻掩码的实现。请理解代码，以便稍后正确应用。

In [ ]:
def create_look_ahead_mask(sequence_length):
    """
    返回一个用 1 填充的下三角矩阵
    
    参数：
        sequence_length -- 矩阵大小
    
    返回：
        mask -- (size, size) 张量
    """
    mask = tf.linalg.band_part(tf.ones((1, sequence_length, sequence_length)), -1, 0)
    return mask 

In [ ]:
x = tf.random.uniform((1, 3))
temp = create_look_ahead_mask(x.shape[1])
temp

<a name='3'></a>
## 3 - 自注意力

正如 Transformer 论文的作者所说，“注意力就是你所需要的”。 

<img src="self-attention.png" alt="Encoder" width="600"/>
<caption><center><font color='purple'><b>图1：Self-Attention计算可视化</font></center></caption>
    
自注意力能够并行处理序列中的各个位置，从而提高训练效率。下面将实现**缩放点积注意力**，输入为查询、键、值和可选掩码，输出为融入上下文信息的向量表示。其数学形式为：
$$
\text { Attention }(Q, K, V)=\operatorname{softmax}\left(\frac{Q K^{T}}{\sqrt{d_{k}}}+{M}\right) V\tag{4}\
$$

* $Q$ 是查询矩阵 
* $K$ 是键矩阵
* $V$ 是值矩阵
* $M$ 是可选的掩码。
* ${d_k}$ 是键向量的维度。缩放点积可避免数值过大，使 softmax 的计算更加稳定。

<a name='ex-3'></a>
### 练习 3 - scaled_dot_product_attention 

实现函数 `scaled_dot_product_attention()` 以创建基于注意力的表示。

**提醒**：掩码参数可以为 `none`（代码中使用 None），也可以传入填充掩码或前瞻掩码。
    
    在应用 softmax 前，将 (1. - mask) 乘以 -1e9，再加到注意力分数上。

**其他提示**
* 你可能会发现 [tf.matmul](https://www.tensorflow.org/api_docs/python/tf/linalg/matmul) 对于矩阵乘法很有用。

In [ ]:
# UNQ_C3 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION scaled_dot_product_attention
def scaled_dot_product_attention(q, k, v, mask):
    """
    计算注意力权重。
      q、k、v 必须具有匹配的主尺寸。
      k, v 必须具有匹配的倒数第二个维度，即：seq_len_k = seq_len_v。
      掩码的形状取决于其类型（填充掩码或前瞻掩码）。
      但它必须是可广播的才能添加。

    参数：
        q -- 查询形状 == (..., seq_len_q, 深度)
        k -- 键形状 == (..., seq_len_k, 深度)
        v -- 值形状 == (..., seq_len_v, depth_v)
        mask: 形状可广播的浮点张量 
              至 (..., seq_len_q, seq_len_k)。默认为无。

    返回：
        输出 -- attention_weights
    """
    # START CODE HERE
    
    matmul_qk = None  # (..., seq_len_q, seq_len_k)

    # 刻度 matmul_qk
    dk = None
    scaled_attention_logits = None

    # 将掩码添加到缩放后的张量中。
    if mask is not None: # 不要替换这个无
        scaled_attention_logits += None 

    # softmax 是最后一个轴 (seq_len_k) 上的归一化后的，因此分数
    # 加起来为 1。
    attention_weights = None  # (..., seq_len_q, seq_len_k)

    output = None  # (..., seq_len_q, depth_v)
    
    # END CODE HERE

    return output, attention_weights

In [ ]:
# 单元测试
scaled_dot_product_attention_test(scaled_dot_product_attention)

干得好！你现在可以实现自注意力。这样，你就可以开始构建编码器块了！ 

<a name='4'></a>
## 4 - 编码器

Transformer 编码器层结合多头自注意力和逐位置前馈神经网络，并将编码结果传递给解码器，供其构造 K 和 V。本节将通过组合多头注意力与前馈网络实现编码器（图 2a）。
<img src="encoder_layer.png" alt="Encoder" width="400"/>
<caption><center><font color='purple'><b>图2a：Transformer编码器层</font></center></caption>

* `MultiHeadAttention` 你可以认为是多次计算自注意力以检测不同的特征。 
* 前馈神经网络包含两个全连接层，我们将其实现为函数 `FullyConnected`

你的输入句子首先通过*多头注意力层*，编码器在编码特定单词时会查看输入句子中的其他单词。然后，多头注意力层的输出被馈送到*前馈神经网络*。完全相同的前馈网络独立地应用于每个位置。
   
* 对于 `MultiHeadAttention` 层，你将使用 [Keras 实现](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MultiHeadAttention)。如果你好奇如何将查询矩阵 Q、键矩阵 K 和值矩阵 V 拆分到不同的头中，可以查看实现。 
* 你还将使用具有两个全连接层的 [Sequential API](https://keras.io/api/models/sequential/) 来构建前馈神经网络层。
    
**注意：** 在Python中，`__call__`方法允许你调用像函数这样的对象。 TensorFlow 通过在 Keras 层内提供 `call` 函数来利用这一点。  这意味着你可以使用像 `a_Layer(arg)` 这样的层，而 TensorFlow 将在内部执行 `a_Layer.call(arg)` 来处理你的数据。  这种设计使得在 TensorFlow 模型中应用层变得直观。    

In [ ]:
def FullyConnected(embedding_dim, fully_connected_dim):
    return tf.keras.Sequential([
        tf.keras.layers.Dense(fully_connected_dim, activation='relu'),  # (batch_size, seq_len, dff)
        tf.keras.layers.Dense(embedding_dim)  # (batch_size, seq_len, embedding_dim)
    ])

<a name='4-1'></a>
### 4.1 编码器层

现在你可以在编码器层中将多头注意力和前馈神经网络配对！你还将使用残差连接和层归一化来帮助加快训练速度（图 2a）。

<a name='ex-4'></a>
### 练习 4 - EncoderLayer

使用 `call()` 方法实现 `EncoderLayer()`

在本练习中，你将使用 `call()` 方法实现一个编码器块（图 2）。 函数应执行以下步骤： 
1. 将 Q、V、K 矩阵和布尔掩码传入多头注意力层。计算*自注意力*时，Q、V 和 K 的输入应相同。为 `return_attention_scores` 和 `training` 设置默认值；训练时，该多头注意力层还会应用 Dropout。
2. 现在通过添加原始输入 `x` 和多头注意力层的输出来添加跳跃连接。 
3. 添加跳跃连接后，将输出传递到第一个归一化层。
4. 最后，重复步骤 1-3，但使用带有 Dropout 层而不是多头注意力层的前馈神经网络。 

<details>
  <summary><font size="2" color="darkgreen"><b>其他提示（点击展开）</b></font></summary>
    
* `__init__` 方法创建 `call` 方法将访问的所有层。无论你想在何处使用 `__init__` 方法中定义的层，都必须使用语法 `self.[insert layer name]`。 
* 可以参考 [MultiHeadAttention](https://www.tensorflow.org/api_docs/python/tf/keras/layers/MultiHeadAttention) 的文档。*注意：若 query、key 和 value 的输入相同，该函数执行的就是自注意力。*
* `self.mha` 的调用参数如下，其中 B 表示批量大小，T 表示查询序列长度，S 表示键和值序列的长度：
 - `query`：形状（B，T，dim）的查询张量。
 - `value`：形状为 (B, S, dim) 的值张量。
 - `key`：可选的键张量，形状为 (B, S, dim)。若未提供，则将 value 同时用作 key 和 value，这是最常见的用法。
 - `attention_mask`：形状为 (B, T, S) 的布尔掩码，指定每个查询位置可以关注哪些键位置：1 表示允许，0 表示禁止。缺省的批量维度和注意力头维度可以通过广播补齐。
 - `return_attention_scores`：布尔值。为 True 时，返回 (attention_output, attention_scores)；为 False 时，仅返回 attention_output。默认值为 False。
 - `training`：Python 布尔值，指示层是否应在训练模式（添加 Dropout）或推理模式（无 Dropout）下运行。默认为使用父层/模型的训练模式，如果没有父层，则默认为 False（推理）。查看 [tf.keras.layers.Dropout](https://www.tensorflow.org/versions/r2.4/api_docs/python/tf/keras/layers/Dropout) 了解更多详细信息（[Keras 常见问题解答](https://keras.io/getting_started/faq/#whats-the-difference-between-the-training-argument-in-call-and-the-trainable-attribute) 中的附加阅读）

In [ ]:
# UNQ_C4 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION EncoderLayer
class EncoderLayer(tf.keras.layers.Layer):
    """
    编码器层由一个多头自注意力子层
    和一个逐位置的全连接前馈子层组成。
    每个子层均使用残差连接，
    然后进行层归一化。
    """
    def __init__(self, embedding_dim, num_heads, fully_connected_dim,
                 dropout_rate=0.1, layernorm_eps=1e-6):
        super(EncoderLayer, self).__init__()

        self.mha = MultiHeadAttention(num_heads=num_heads,
                                      key_dim=embedding_dim,
                                      dropout=dropout_rate)

        self.ffn = FullyConnected(embedding_dim=embedding_dim,
                                  fully_connected_dim=fully_connected_dim)

        self.layernorm1 = LayerNormalization(epsilon=layernorm_eps)
        self.layernorm2 = LayerNormalization(epsilon=layernorm_eps)

        self.dropout_ffn = Dropout(dropout_rate)
    
    def call(self, x, training, mask):
        """
        编码器层的前向传播
        
        参数：
            x -- 形状为 (batch_size, input_seq_len, embedding_dim) 的张量
            training -- 布尔值；设为 True 时，
                        启用 Dropout 层的训练模式
            mask -- 布尔掩码，用于屏蔽填充位置，
                    防止其参与注意力计算
        返回：
            encoder_layer_out -- 形状为 (batch_size, input_seq_len, embedding_dim) 的张量
        """
        # START CODE HERE
        # 使用 mha(~1 line) 计算自注意力。
        # 如果训练期间 Dropout 参数非零，Dropout 会自动由 Keras 添加
        self_mha_output = None  # 自我关注(batch_size, input_seq_len, embedding_dim)
        
        # 跳跃连接
        # 将归一化层应用于输入和注意力输出的总和以获得  
        # 多头注意力层的输出（~1 行）
        skip_x_attention = None  # (batch_size, input_seq_len, embedding_dim)

        # 通过 ffn （~1 行）传递多头注意力层的输出
        ffn_output = None  # (batch_size, input_seq_len, embedding_dim)
        
        # 在训练期间将 Dropout 层应用于 ffn 输出（~1 行）
        # 使用`training=training` 
        ffn_output = None
        
        # 将归一化层应用于多头注意力输出（跳跃连接）和 ffn 输出的总和，以获得
        # 编码器层的输出（~1 行）
        encoder_layer_out = None  # (batch_size, input_seq_len, embedding_dim)
        # END CODE HERE
        
        return encoder_layer_out
    

In [ ]:
# 单元测试
EncoderLayer_test(EncoderLayer)

<a name='4-2'></a>
### 4.2 - 完整编码器

干得好！你现在已经成功实现了位置编码、自注意力和编码器层 - 给自己一个鼓励吧。现在你已准备好构建完整的 Transformer 编码器（图 2b），你将在其中嵌入输入并添加你计算的位置编码。然后，你将编码的嵌入提供给一堆编码器层。 

<img src="encoder.png" alt="Encoder" width="330"/>
<caption><center><font color='purple'><b>图 2b：Transformer 编码器</font></center></caption>


<a name='ex-5'></a>
### 练习 5 - Encoder

使用 `call()` 方法完成 `Encoder()` 函数来嵌入输入、添加位置编码并实现多个编码器层。 

在本练习中，你将使用嵌入层、位置编码和多个 EncoderLayer 来初始化编码器。你的 `call()` 方法将执行以下步骤： 
1. 将你的输入传递到嵌入层。
2. 通过将其乘以嵌入尺寸的平方根来缩放嵌入。请记住在计算平方根之前将嵌入维度转换为数据类型 `tf.float32`。
3. 将位置编码：self.pos_encoding `[:, :seq_len, :]` 添加到嵌入中。
4. 将编码嵌入传递到 Dropout 层，记住使用 `training` 参数来设置模型训练模式。 
5. 使用 for 循环将 Dropout 层的输出传递到编码层堆栈。

In [ ]:
 # UNQ_C5 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION
class Encoder(tf.keras.layers.Layer):
    """
    完整编码器先将输入传入嵌入层，
    再加入位置编码，
    然后将结果传入堆叠的编码器层。
        
    """  
    def __init__(self, num_layers, embedding_dim, num_heads, fully_connected_dim, input_vocab_size,
               maximum_position_encoding, dropout_rate=0.1, layernorm_eps=1e-6):
        super(Encoder, self).__init__()

        self.embedding_dim = embedding_dim
        self.num_layers = num_layers

        self.embedding = Embedding(input_vocab_size, self.embedding_dim)
        self.pos_encoding = positional_encoding(maximum_position_encoding, 
                                                self.embedding_dim)


        self.enc_layers = [EncoderLayer(embedding_dim=self.embedding_dim,
                                        num_heads=num_heads,
                                        fully_connected_dim=fully_connected_dim,
                                        dropout_rate=dropout_rate,
                                        layernorm_eps=layernorm_eps) 
                           for _ in range(self.num_layers)]

        self.dropout = Dropout(dropout_rate)
        
    def call(self, x, training, mask):
        """
        编码器的前向传播
        
        参数：
            x -- 形状为 (batch_size, input_seq_len) 的张量
            training -- 布尔值；设为 True 时，
                        启用 Dropout 层的训练模式
            mask -- 布尔掩码，用于屏蔽填充位置，
                    防止其参与注意力计算
        返回：
            x -- 形状为 (batch_size, input_seq_len, embedding_dim) 的张量
        """
        seq_len = tf.shape(x)[1]
        
        # START CODE HERE
        # 通过嵌入层传递输入
        x = None  # (batch_size, input_seq_len, embedding_dim)
        # 通过乘以嵌入维度的平方根来缩放嵌入
        x *= None
        # 将位置编码添加到嵌入中
        x += None
        # 通过 Dropout 层传递编码嵌入
        # 使用`training=training`
        x = None
        # 通过编码层堆栈传递输出 
        for i in range(self.num_layers):
            x = None
        # END CODE HERE

        return x  # (batch_size, input_seq_len, embedding_dim)

In [ ]:
# 单元测试    
Encoder_test(Encoder)

<a name='5'></a>
## 5 - 解码器

解码器层采用编码器生成的 K 和 V 矩阵，并使用输出中的 Q 矩阵计算第二个多头注意力层（图 3a）。

<img src="decoder_layer.png" alt="Encoder" width="250"/>
<caption><center><font color='purple'><b>图3a：Transformer解码器层</font></center></caption>

<a name='5-1'></a>    
### 5.1 - 解码器层
同样，你将把多头注意力与前馈神经网络配对，但这次你将实现两个多头注意力层。你还将使用残差连接和层归一化来帮助加快训练速度（图 3a）。

<a name='ex-6'></a>    
### 练习 6 - DecoderLayer
    
使用 `call()` 方法实现 `DecoderLayer()`
    
1. 块 1 是一个多头注意力层，具有残差连接和前瞻掩码。与 `EncoderLayer` 一样，Dropout 是在多头注意力层中定义的。
2. 块 2 将考虑编码器的输出，因此多头注意力层将从编码器接收 K 和 V，并从块 1 接收 Q。然后，你将应用归一化层和残差连接，就像之前对 `EncoderLayer` 所做的那样。
3. 最后，块 3 是一个前馈神经网络，具有 Dropout 和归一化层以及残差连接。
    
**其他提示：**
* 前两个块与 EncoderLayer 非常相似，只是在计算 self-attention 时将返回 `attention_scores`

In [ ]:
# UNQ_C6 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION DecoderLayer
class DecoderLayer(tf.keras.layers.Layer):
    """
    解码器层包含两个多头注意力子层：
    第一个对目标序列执行带掩码的自注意力，
    第二个结合编码器输出执行交叉注意力，
    最后接一个逐位置的全连接前馈子层。
    """
    def __init__(self, embedding_dim, num_heads, fully_connected_dim, dropout_rate=0.1, layernorm_eps=1e-6):
        super(DecoderLayer, self).__init__()

        self.mha1 = MultiHeadAttention(num_heads=num_heads,
                                      key_dim=embedding_dim,
                                      dropout=dropout_rate)

        self.mha2 = MultiHeadAttention(num_heads=num_heads,
                                      key_dim=embedding_dim,
                                      dropout=dropout_rate)

        self.ffn = FullyConnected(embedding_dim=embedding_dim,
                                  fully_connected_dim=fully_connected_dim)

        self.layernorm1 = LayerNormalization(epsilon=layernorm_eps)
        self.layernorm2 = LayerNormalization(epsilon=layernorm_eps)
        self.layernorm3 = LayerNormalization(epsilon=layernorm_eps)

        self.dropout_ffn = Dropout(dropout_rate)
    
    def call(self, x, enc_output, training, look_ahead_mask, padding_mask):
        """
        解码器层的前向传播
        
        参数：
            x -- 形状为 (batch_size, target_seq_len, embedding_dim) 的张量
            enc_output --  形状为 (batch_size, input_seq_len, embedding_dim) 的张量
            training -- 布尔值；设为 True 时，
                        启用 Dropout 层的训练模式
            look_ahead_mask -- target_input 的布尔掩码
            padding_mask -- 第二个多头注意力层的布尔掩码
        返回：
            out3 -- 形状为 (batch_size, target_seq_len, embedding_dim) 的张量
            attn_weights_block1 -- 形状为 (batch_size, num_heads, target_seq_len, target_seq_len) 的张量
            attn_weights_block2 -- 形状为 (batch_size, num_heads, target_seq_len, input_seq_len) 的张量
        """
        
        # START CODE HERE
        # enc_output.shape == (batch_size, input_seq_len, embedding_dim)
        
        # 区块 1
        # 计算自注意力并返回注意力分数为 attn_weights_block1。
        # Dropout 将在训练期间应用（~1 行）。
        mult_attn_out1, attn_weights_block1 = self.mha1(None, None, None, None, return_attention_scores=True)  # (batch_size, target_seq_len, embedding_dim)
        
        # 将层归一化 (layernorm1) 应用到注意输出和输入的总和（~1 行）
        Q1 = None

        # 区块 2
        # 使用第一个块的 Q 和编码器输出的 K 和 V 计算自注意力。 
        # Dropout 将在训练期间应用
        # 返回注意力分数为 attn_weights_block2（~1 行） 
        mult_attn_out2, attn_weights_block2 = self.mha2(None, None, None, None, return_attention_scores=True)  # (batch_size, target_seq_len, embedding_dim)
        
        # 将层归一化 (layernorm2) 应用到注意力输出与第一个块的输出之和（~1 行）
        mult_attn_out2 = None  # (batch_size, target_seq_len, embedding_dim)
                
        #区块 3
        # 通过 ffn 传递第二个块的输出
        ffn_output = None  # (batch_size, target_seq_len, embedding_dim)
        
        # 将 Dropout 层应用于 ffn 输出
        # 使用`training=training`
        ffn_output = None
        
        # 将层归一化 (layernorm3) 应用到 ffn 输出与第二个块的输出之和
        out3 = None  # (batch_size, target_seq_len, embedding_dim)
        # END CODE HERE

        return out3, attn_weights_block1, attn_weights_block2
    

In [ ]:
# 单元测试
DecoderLayer_test(DecoderLayer, create_look_ahead_mask)

<a name='5-2'></a> 
### 5.2 - 完整解码器
你就快到了！是时候使用解码器层构建完整的 Transformer 解码器了（图 3b）。你将嵌入输出并添加位置编码。然后，你将编码的嵌入提供给解码器层堆栈。 


<img src="decoder.png" alt="Encoder" width="300"/>
<caption><center><font color='purple'><b>图3b：Transformer解码器</font></center></caption>

<a name='ex-7'></a>     
### 练习 7 - Decoder

使用 `call()` 方法实现 `Decoder()` 来嵌入输出、添加位置编码并实现多个解码器层。
 
在本练习中，你将使用嵌入层、位置编码和多个 DecoderLayer 来初始化解码器。你的 `call()` 方法将执行以下步骤： 
1. 将生成的输出传递到嵌入层。
2. 通过将其乘以嵌入尺寸的平方根来缩放嵌入。请记住在计算平方根之前将嵌入维度转换为数据类型 `tf.float32`。
3. 将位置编码：self.pos_encoding `[:, :seq_len, :]` 添加到嵌入中。
4. 将编码嵌入传递到 Dropout 层，记住使用 `training` 参数来设置模型训练模式。 
5. 使用 for 循环将 Dropout 层的输出传递到解码层堆栈。

In [ ]:
# UNQ_C7 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION Decoder
class Decoder(tf.keras.layers.Layer):
    """
    完整解码器先将目标序列输入嵌入层，
    再加入位置编码，
    然后将结果传入堆叠的解码器层。
        
    """ 
    def __init__(self, num_layers, embedding_dim, num_heads, fully_connected_dim, target_vocab_size,
               maximum_position_encoding, dropout_rate=0.1, layernorm_eps=1e-6):
        super(Decoder, self).__init__()

        self.embedding_dim = embedding_dim
        self.num_layers = num_layers

        self.embedding = Embedding(target_vocab_size, self.embedding_dim)
        self.pos_encoding = positional_encoding(maximum_position_encoding, self.embedding_dim)

        self.dec_layers = [DecoderLayer(embedding_dim=self.embedding_dim,
                                        num_heads=num_heads,
                                        fully_connected_dim=fully_connected_dim,
                                        dropout_rate=dropout_rate,
                                        layernorm_eps=layernorm_eps) 
                           for _ in range(self.num_layers)]
        self.dropout = Dropout(dropout_rate)
    
    def call(self, x, enc_output, training, 
           look_ahead_mask, padding_mask):
        """
        解码器的前向传播
        
        参数：
            x -- 形状为 (batch_size, target_seq_len) 的词索引张量
            enc_output --  形状为 (batch_size, input_seq_len, embedding_dim) 的张量
            training -- 布尔值；设为 True 时，
                        启用 Dropout 层的训练模式
            look_ahead_mask -- target_input 的布尔掩码
            padding_mask -- 第二个多头注意力层的布尔掩码
        返回：
            x -- 形状为 (batch_size, target_seq_len, embedding_dim) 的张量
            attention_weights - 字典，保存各解码器层的注意力权重张量
                                形状分别为 (batch_size, num_heads, target_seq_len, target_seq_len) 和 (batch_size, num_heads, target_seq_len, input_seq_len)
        """

        seq_len = tf.shape(x)[1]
        attention_weights = {}
        
        # START CODE HERE
        # 创建词嵌入 
        x = None  # (batch_size, target_seq_len, embedding_dim)
        
        # 通过乘以嵌入维度的平方根来缩放嵌入
        x *= None
        
        # 计算位置编码并添加到词嵌入
        x += None

        # 将 Dropout 层应用于 x
        # 使用`training=training`
        x = None

        # 使用 for 循环将 x 通过解码器层堆栈并更新 attention_weights （总共约 4 行）
        for i in range(self.num_layers):
            # 将 x 和编码器输出传递给解码器层堆栈并保存注意力权重第 1 块和第 2 块的
            # （~1 行）
            x, block1, block2 = self.dec_layers[i](None, None, None,
                                                 None, None)

            #用块 1 和块 2 的注意力权重更新 attention_weights 字典
            attention_weights['decoder_layer{}_block1_self_att'.format(i+1)] = None
            attention_weights['decoder_layer{}_block2_decenc_att'.format(i+1)] = None
        # END CODE HERE
        
        # x.shape == (batch_size, target_seq_len, embedding_dim)
        return x, attention_weights

In [ ]:
# 单元测试
Decoder_test(Decoder, create_look_ahead_mask, create_padding_mask)

<a name='6'></a> 
## 6 - Transformer

你已经完成了本次作业的各个组件，也来到了深度学习专项课程的最后一道练习。现在，将它们组合为完整的 Transformer 模型！

<img src="transformer.png" alt="Transformer" width="550"/>
<caption><center><font color='purple'><b>图4：Transformer</font></center></caption>
    
通过 Transformer 架构的数据流如下：
* 首先，你的输入通过编码器，它只是你实现的重复编码器层：
    - 输入的嵌入和位置编码
    - 对输入应用多头自注意力
    - 前馈神经网络帮助检测特征
* 然后预测的输出通过解码器，由你实现的解码器层组成：
    - 输出的嵌入和位置编码
    - 对已生成的输出应用带掩码的多头自注意力
    - 多头注意力，其中 Q 来自第一个多头注意力层，K 和 V 来自编码器
    - 帮助检测特征的前馈神经网络
* 最后，在第 N 个解码器层之后，应用一个全连接层和一个 softmax 来生成序列中下一个输出的预测。

<a name='ex-8'></a> 
### 练习 8 - Transformer

使用 `call()` 方法实现 `Transformer()`
1. 使用适当的掩码将输入传递到编码器。
2. 使用适当的掩码将编码器输出和目标通过解码器。
3. 应用线性变换和 softmax 来获得预测。

In [ ]:
# UNQ_C8 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION Transformer
class Transformer(tf.keras.Model):
    """
    组合编码器与解码器，构建完整的 Transformer
    """
    def __init__(self, num_layers, embedding_dim, num_heads, fully_connected_dim, input_vocab_size, 
               target_vocab_size, max_positional_encoding_input,
               max_positional_encoding_target, dropout_rate=0.1, layernorm_eps=1e-6):
        super(Transformer, self).__init__()

        self.encoder = Encoder(num_layers=num_layers,
                               embedding_dim=embedding_dim,
                               num_heads=num_heads,
                               fully_connected_dim=fully_connected_dim,
                               input_vocab_size=input_vocab_size,
                               maximum_position_encoding=max_positional_encoding_input,
                               dropout_rate=dropout_rate,
                               layernorm_eps=layernorm_eps)

        self.decoder = Decoder(num_layers=num_layers, 
                               embedding_dim=embedding_dim,
                               num_heads=num_heads,
                               fully_connected_dim=fully_connected_dim,
                               target_vocab_size=target_vocab_size, 
                               maximum_position_encoding=max_positional_encoding_target,
                               dropout_rate=dropout_rate,
                               layernorm_eps=layernorm_eps)

        self.final_layer = Dense(target_vocab_size, activation='softmax')
    
    def call(self, input_sentence, output_sentence, training, enc_padding_mask, look_ahead_mask, dec_padding_mask):
        """
        完整 Transformer 的前向传播
        参数：
            input_sentence -- 形状为 (batch_size, input_seq_len) 的张量，
                              保存输入句子的词索引
            output_sentence -- 形状为 (batch_size, target_seq_len) 的张量，
                               保存目标输出句子的词索引
            training -- 布尔值；设为 True 时，
                        启用 Dropout 层的训练模式
            enc_padding_mask -- 布尔掩码，用于屏蔽输入中的填充位置，
                                防止其参与注意力计算
            look_ahead_mask -- target_input 的布尔掩码
            dec_padding_mask -- 第二个多头注意力层的布尔掩码
        返回：
            final_output -- 形状为 (batch_size, target_seq_len, target_vocab_size) 的词汇概率张量
            attention_weights - 字典，保存各解码器层的注意力权重张量
                                形状分别为 (batch_size, num_heads, target_seq_len, target_seq_len) 和 (batch_size, num_heads, target_seq_len, input_seq_len)
        
        """
        # START CODE HERE
        # 使用适当的参数调用 self.encoder 以获取编码器输出
        enc_output = None  # (batch_size, inp_seq_len, embedding_dim)
        
        # 使用适当的参数调用 self.decoder 以获取解码器输出
        # dec_output.shape == (batch_size, tar_seq_len, embedding_dim)
        dec_output, attention_weights = self.decoder(None, None, None, None, None)
        
        # 通过线性层和 softmax （~2 行）传递解码器输出
        final_output = None # (batch_size, tar_seq_len, target_vocab_size)
        # END CODE HERE

        return final_output, attention_weights

In [ ]:
# 单元测试
Transformer_test(Transformer, create_look_ahead_mask, create_padding_mask)

## 结论

你已完成作业的计分部分。到目前为止，你已经： 

* 创建位置编码来捕获数据中的顺序关系
* 使用词嵌入计算的缩放点积自注意力
* 实现带掩码的多头注意力
* 构建并训练 Transformer 模型

<font color='blue'>
    <b>你应该记住什么</b>：

- 自注意力与逐位置前馈网络的组合支持并行计算，从而实现*更快的训练*。
- 使用生成的查询 Q、键 K 和值 V 矩阵计算自注意力。
- 在词嵌入中添加位置编码是在自注意力计算中包含序列信息的有效方法。 
- 多头注意力可以帮助检测句子中的多个特征。
- 掩码可以阻止模型在训练时访问未来位置的信息，也可以防止模型将注意力分配给用于补齐序列的 0。

现在你已经完成了 Transformer 作业，请务必查看不计分的实验室，将 Transformer 模型应用于实际用例，例如名称实体识别 (NER) 和问答 (QA)。  


# 恭喜你完成深度学习专项课程！！！！！！ 🎉🎉🎉🎉🎉

这是专项课程的最后一个评分作业。现在是时候庆祝你的辛勤工作和奉献精神了！ 

<a name='7'></a> 
## 7 - 参考文献

Transformer 算法是 Vaswani 等人提出的。 （2017）。 

- Ashish Vaswani、Noam Shazeer、Niki Parmar、Jakob Uszkoreit、Llion Jones、Aidan N. Gomez、Lukasz Kaiser、Illia Polosukhin (2017)。 [注意力就是你所需要的](https://arxiv.org/abs/1706.03762) 